# 01 — Data Exploration & EDA
## DermaVision DL · HAM10000 / ISIC 2018 Task 3 Dataset

**Goals:**
1. Understand class distribution and identify imbalance
2. Visually inspect sample images per class
3. Analyse image dimensions and quality
4. Visualise feature space via t-SNE
5. Compute class weights for training
6. Document findings and derive preprocessing decisions

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Paths
NOTEBOOK_DIR = Path(os.path.abspath(''))
REPO_ROOT    = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR     = REPO_ROOT / 'data' / 'HAM10000'
IMG_DIR      = DATA_DIR / 'ISIC2018_Task3_Training_Input'
META_PATH    = DATA_DIR / 'ISIC2018_Task3_Training_GroundTruth' / 'ISIC2018_Task3_Training_GroundTruth.csv'

print('CWD            :', NOTEBOOK_DIR)
print('Repo root      :', REPO_ROOT)
print('Images exist   :', IMG_DIR.exists())
print('Metadata exists:', META_PATH.exists())
if IMG_DIR.exists():
    print('Image count    :', len(list(IMG_DIR.glob('*.jpg'))))

## 1 · Load Metadata

In [ ]:
df_raw = pd.read_csv(META_PATH)
print(f'Total samples : {len(df_raw):,}')
print(f'Columns       : {list(df_raw.columns)}')
df_raw.head()

In [ ]:
LABEL_MAP = {
    'MEL'  : 'Melanoma',
    'NV'   : 'Melanocytic Nevi',
    'BCC'  : 'Basal Cell Carcinoma',
    'AKIEC': 'Actinic Keratosis',
    'BKL'  : 'Benign Keratosis',
    'DF'   : 'Dermatofibroma',
    'VASC' : 'Vascular Lesion'
}

label_cols       = list(LABEL_MAP.keys())
df               = df_raw.copy()
df['dx']         = df[label_cols].idxmax(axis=1)
df['label_name'] = df['dx'].map(LABEL_MAP)
df['image_path'] = df['image'].apply(lambda x: str(IMG_DIR / f'{x}.jpg'))

missing = df[~df['image_path'].apply(lambda p: Path(p).exists())]
print(f'Total samples  : {len(df):,}')
print(f'Missing images : {len(missing)}')
df[['image','dx','label_name']].head(3)

## 2 · Class Distribution

A critical first step is understanding **how balanced** the dataset is.
Severe imbalance requires corrective strategies (weighted loss, oversampling).

In [ ]:
class_counts = df['label_name'].value_counts()
class_pct    = (class_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(class_counts.index, class_counts.values,
                   color=sns.color_palette('muted', len(class_counts)))
axes[0].set_title('Class Distribution (absolute counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lesion Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35)
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', va='bottom', fontsize=9)
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=sns.color_palette('muted', len(class_counts)), startangle=140)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_class_distribution.png', bbox_inches='tight')
plt.show()
print('\nClass breakdown:')
for cls, cnt, pct in zip(class_counts.index, class_counts.values, class_pct.values):
    print(f'  {cls:<25} {cnt:>5}  ({pct}%)')

### 🔍 Finding
**Melanocytic Nevi (NV)** dominates the dataset at ~67% of all samples.
The rarest classes (Vascular Lesion, Dermatofibroma) represent < 2% each.
This severe **class imbalance** means:
- A naive model achieves ~67% accuracy by always predicting NV — accuracy is **not** the right metric
- We will use **weighted cross-entropy loss** (weights inversely proportional to frequency)
- Primary metrics: **AUC-ROC**, **Sensitivity**, **Specificity** per class

## 3 · Sample Images per Class

In [ ]:
N_SAMPLES = 5
fig, axes = plt.subplots(len(LABEL_MAP), N_SAMPLES,
                         figsize=(N_SAMPLES * 2.5, len(LABEL_MAP) * 2.5))
fig.suptitle('Sample Images per Lesion Class', fontsize=14, fontweight='bold', y=1.01)
for row, cls in enumerate(LABEL_MAP.keys()):
    samples = df[df['dx'] == cls].sample(N_SAMPLES, random_state=42)
    for col, (_, row_data) in enumerate(samples.iterrows()):
        img = Image.open(row_data['image_path'])
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(LABEL_MAP[cls], fontsize=8, fontweight='bold',
                                      rotation=0, labelpad=110, va='center')
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_sample_images.png', bbox_inches='tight')
plt.show()

## 4 · Image Dimension & Quality Analysis

In [ ]:
widths, heights, corrupt = [], [], []
for path in df['image_path']:
    try:
        with Image.open(path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except Exception:
        corrupt.append(path)
print(f'Corrupt / unreadable images : {len(corrupt)}')
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[1].hist(heights, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_image_sizes.png', bbox_inches='tight')
plt.show()

### 🔍 Finding
All images are **600×450 px** — uniform resolution, no resizing inconsistencies.
No corrupt images found.
For training we will resize to **224×224** (EfficientNet-B3 input) using `albumentations.Resize`.

## 5 · t-SNE Feature Space Visualisation

Colour histogram features extracted per image to visualise class separability
in 2D **before** any deep learning.

In [ ]:
from sklearn.manifold import TSNE
import cv2

df_sample = df.groupby('dx', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 100), random_state=42)
)
features = []
for path in df_sample['image_path']:
    img  = cv2.imread(path)
    img  = cv2.resize(img, (64, 64))
    hist = cv2.calcHist([img], [0,1,2], None, [8,8,8], [0,256,0,256,0,256])
    features.append(hist.flatten())
features  = np.array(features, dtype=np.float32)
features /= features.sum(axis=1, keepdims=True) + 1e-8
print(f'Running t-SNE on {len(features)} samples...')
tsne   = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeds = tsne.fit_transform(features)
print('Done.')

In [ ]:
palette    = sns.color_palette('tab10', n_colors=7)
label_list = df_sample['dx'].tolist()
fig, ax = plt.subplots(figsize=(10, 7))
for i, cls in enumerate(LABEL_MAP.keys()):
    mask = np.array([l == cls for l in label_list])
    ax.scatter(embeds[mask, 0], embeds[mask, 1],
               label=LABEL_MAP[cls], alpha=0.65, s=18, color=palette[i])
ax.legend(loc='best', fontsize=8, markerscale=2)
ax.set_title('t-SNE of Colour Histogram Features (pre-DL)', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_tsne.png', bbox_inches='tight')
plt.show()

### 🔍 Finding
Low-level colour features produce **significant class overlap** in the t-SNE projection.
Simple colour statistics are **insufficient** for discrimination —
justifying a deep convolutional backbone to learn hierarchical representations.

## 6 · Class Weight Computation

Pre-computed and imported directly into the training pipeline.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes_arr   = np.array(sorted(df['dx'].unique()))
weights       = compute_class_weight('balanced', classes=classes_arr, y=df['dx'].values)
class_weights = dict(zip(classes_arr, weights.round(4)))

print('Class weights (balanced):')
for cls, w in sorted(class_weights.items(), key=lambda x: -x[1]):
    print(f'  {cls:<8} ({LABEL_MAP[cls]:<25})  weight = {w:.4f}')

### 🔍 Finding
Rare classes receive weights ~10–20× higher than the dominant **Melanocytic Nevi** class.
These weights will be passed to `torch.nn.CrossEntropyLoss(weight=...)` in all experiments.

## 7 · EDA Summary & Decisions

| Finding | Decision |
|---|---|
| Severe class imbalance (NV ≈ 67%) | Weighted cross-entropy loss in all experiments |
| All images 600×450 px, no corruption | Resize to 224×224 for EfficientNet-B3 |
| Colour features insufficient (t-SNE overlap) | Deep backbone required — transfer learning justified |
| No patient demographic metadata available | Document as ethical limitation |
| Dataset predominantly European skin tones | Document as ethical limitation |

→ Proceed to `02_model_training.ipynb`